In [1]:
import os
import time
import re

import pandas as pd
import requests
import urllib.parse
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

from Constants import Constants as const

In [21]:
# === 1. 初始化 Selenium 浏览器 ===
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager(driver_version="138").install()), options=options)

# 用 JS 覆盖 navigator.webdriver
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
  "source": """
    Object.defineProperty(navigator, 'webdriver', {
      get: () => undefined
    })
  """
})

{'identifier': '2'}

In [6]:
gvkey_glassdoor_url = pd.read_excel(os.path.join(const.TEMP_PATH, '20250625_ue_with_glassdoor_web_fill_miss.xlsx'))

In [22]:
driver.get('https://www.glassdoor.com/index.htm')

In [18]:
driver.find_element(by=By.ID, value='inlineUserEmail').send_keys('wangyouan@gmail.com')
driver.find_element(by=By.XPATH, value='//*[@id="InlineLoginModule"]/div/div[1]/div/div/div/div/form/div[2]/div/button').click()


In [19]:
driver.find_element(by=By.ID, value='inlineUserPassword').send_keys('4Pa5EjPghdTV-c5')
driver.find_element(by=By.XPATH, value='//*[@id="InlineLoginModule"]/div/div[1]/div/div/div/div/form/div[2]/div/button').click()

In [23]:
driver.close()

In [26]:
def slugify(s):
    # Glassdoor 评论页中的 shortName 通常无需处理，但保险起见进行标准清洗
    s = re.sub(r'[^\w\s-]', '', s)
    s = re.sub(r'\s+', '-', s.strip())
    return s

def get_glassdoor_reviews_url(company_name):
    base_api = "https://www.glassdoor.com/autocomplete/employers?term="
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        query = urllib.parse.quote(company_name)
        res = requests.get(base_api + query, headers=headers)
        if res.status_code == 200:
            data = res.json()
            if data:
                item = data[0]
                cid = item['id']
                short = slugify(item['shortName']) if 'shortName' in item else slugify(item['name'])
                return f"https://www.glassdoor.com/Reviews/{short}-Reviews-E{cid}.htm"
    except Exception as e:
        print(f"[Error] {company_name}: {e}")

    return None


In [28]:
def construct_review_url(company_name, company_id):
    short = slugify(company_name)
    return f"https://www.glassdoor.com/Reviews/{short}-Reviews-E{company_id}.htm"

In [29]:
print(construct_review_url('K-V Pharmaceutical', '2500'))

https://www.glassdoor.com/Reviews/K-V-Pharmaceutical-Reviews-E2500.htm


In [33]:
print('\n'.join([f'"{i}",' for i in '''Metro Inc
OptiCare Health Systems Inc.
Orthovita Inc
POSCO Holdings Inc
Trimac Transportation Ltd
Medco Health Solutions Inc.
Ramp Corp
Penske Truck Leasing Co LP
Weyerhaeuser Company Ltd
EPR Properties
BlueLinx Holdings Inc
Hyco International Inc
Johns Manville Corp
Extendicare Inc
Sevenson Environmental Services Inc
Comcast Cable Communications Inc.
Appalachian Power
Ryerson Tull Inc-Old
KT Corporation
West Marine Inc
Swift Transportation Co
Standard Register Co (The)
Public Service Company of New Hampshire
Regional Brands Inc
Connecticut Light & Power Co
Sure Energy Inc
Loomis Fargo & Co
Garda World Security Corp
Petro USA Inc
Gas Natural Inc
Verizon Inc/PA
Consolidated Container Co LLC
Vanguard Car Rental Group Inc
Tube City IMS Corp
Data Systems Inc
G&K Services Inc
West Energy Ltd
Noble Environmental Power Inc
Chrysler Group LLC
Bea Systems Inc
RAE Systems Inc.
America Service Group Inc
Black Box Corp
D-Box Technologies Inc
Wackenhut Corp
IFCI International Corp
Albertsons Cos Inc
Industrial Services of America Inc
Fresenius Medical Care AG
Pure World Inc
Certainteed Corp
Pennsylvania Power Co
Superior Bancorp
Stratus Services Group Inc
Renal Care Group Inc.
Central Freight Lines Inc
Car Inc
Constar International Inc
Cenveo Inc.
American Medical Security Group Inc
Stratford American Corp
Pathfinder Bancorp Inc
Titan Technologies Inc
Bunzl PLC
Boardwalk Pipelines LLC
LMI Aerospace Inc
MeadWestvaco Corp
Greyhound Lines Inc.
B Communications Ltd
Transportation Component Inc
Cooper Industries Plc
Ohio Power
Wheeling Island Gaming Inc
Dynamic Sciences International
Cushman & Wakefield Ltd
Domino's Inc.
Ferrellgas Partners LP
Arizona Public Service Co
WJ Communications Inc
DIRECTV Holdings LLC
L'Air Liquide SA
CEVA Logistics Inc
Rhodia
Spartech Corp
Tasty Baking Co
Dr. Pepper Bottling Holdings
ABB Ltd
ASI Liquidating Corp
Total Logistics Inc
Sodexho Marriott Services Inc
Community Care Services Inc
National Grid plc
Lafarge North America Inc.
Indiana Michigan Power Co
Energy East Corp.
Omega Protein Corp
Kentucky Power
Remington Arms Co Inc
Rand Logistics Inc
First Aviation Services Inc
BTU International Inc
AG Growth International Inc
Sunoco Logistics Partners LP
Alion Science and Technology Corp
Coleman Cable Inc
Columbus Southern Power Corp
Monongahela Power
Park Aerospace Corp
Maverick Minerals Corp
Commonwealth Edison Co
TFI International Inc
Capital Power Corp
Ohio Edison Co
Sierra Pacific Power Co
Strateco Resources Inc
CompX International Inc.
Inventure Foods Inc
Aquila Resources Inc
Curis Resources Ltd
Regal Entertainment Group
Univar Corp
Metropolitan Edison
Ralcorp Holdings Inc.
Verus International Inc
Pittston Co-Consolidated
Puget Sound Energy Inc
Axalta Coating Systems Ltd
AEP Industries Inc
Southern Power Co
Tecogen Inc
Central Maine Power Co
NES Rentals Holdings Inc
Pivot Technology Solutions Inc
Allegheny Energy Supply Co LLC
Care Capital Properties Inc
Refco Inc
Community Medical Transport
Ensign Energy Services Inc
Telecomunicaciones de Puerto Rico Inc.
Sense Technologies Inc
Dextera Surgical Inc
Skyline Corp
Northwest Pipe Co
Canon Inc
Colt Resources Inc
Cadus Corporation
Associated Materials LLC
SIFCO Industries Inc.
Superior Plus Corp
ICTS International NV
Progressive Waste Solutions Ltd
Exa Corp
Stantec Inc
Texas Mineral Resources Corp
SLR Investment Corp
Florida Power & Light Co
Hawaiian Electric Co
Danaos Corporation
DS Healthcare Group Inc
Opta Minerals Inc
Parks America Inc
Central Hudson Gas & Electric
Westar Energy Inc
International Game Technology PLC
KapStone Paper & Packaging Corp
Fort Dearborn Income Securities Inc.
Tampa Electric Co
Ore Holdings Inc
NBCUniversal Media LLC
Nuverra Environmental Solutions Inc
AdvancePierre Foods Holdings Inc
CSRA Inc
Lifetime Brands Inc
San Diego Gas & Electric Co
Fairmount Santrol Holdings Inc
BPO Management Services Inc
CWC Energy Services Corp
Jorgensen (Earle M.) Co
Eastern Co (The)
Anheuser-Busch InBev SA/NV
Baltimore Gas & Electric Co
Everest Healthcare Services Corp
Air Industries Group Inc
Command Security Corp
Atlantic American Corp
Atlantic City Electric Co
Genesis Healthcare Inc
Kentucky Utilities Co
Boyd Group Services Inc
Consumers Energy Co
SI Financial Group Inc
Land O'Lakes Inc.
Central Steel & Wire Co
GFL Environmental Inc
South Carolina Elec & Gas Co
Blue Apron Holdings Inc
Intercontinental Hotels Group PLC
Highcliff Metals Corp
United Security Bancshares
PECO Energy Co
CH2M HILL Companies Ltd.
Komatsu Ltd
People Corp
Wynn Las Vegas LLC
Duke Energy Carolinas LLC
UGI Utilities Inc
Armstrong Flooring Inc
Sprague Resources LP
Spirent Communications
Mercury Air Group Inc.
Unique Fabricating Inc
Agiliti Inc
KWG Resources Inc
United States Postal Service
Green Thumb Industries Inc
ALR Technologies SG LTD
International Baler Corp
Professional Transportation Group Inc
Max Sound Corp
Wisconsin Public Service Corp
Pason Systems Inc
North Shore Gas Co
Strattec Security Corp
Forterra Inc
Sterigenics International Inc
Carrier Access Corp
7-Eleven Inc
Koppers Inc.
Southwestern Electric Power Co
Oilgear Company (The)
TAT Technologies Ltd
Georgia Power Co
K-Sea Transportation Partners LP
Atlas Corp
Thomson Reuters Corp
Comprehensive Care Corp
RS Technologies Inc
Multiband Corp
Mitec Technologies Inc
China Natural Resources Inc
Dollar Thrifty Automotive Group Inc.
Lafarge SA
Centerline Holding Co
Teva Pharmaceutical Industries Ltd
Oplink Communications Inc
Gateway Energy Corp
Eagle Energy Inc
NSTAR Electric Co
Zale Corp
West Corp
Crude Carriers Corp
CPS Technologies Corp
Aecon Group Inc
Fujitsu Ltd
Silvan Industries Inc
Celera Corp
AmerenEnergy Generating Co'''.split('\n')]))

"Metro Inc",
"OptiCare Health Systems Inc.",
"Orthovita Inc",
"POSCO Holdings Inc",
"Trimac Transportation Ltd",
"Medco Health Solutions Inc.",
"Ramp Corp",
"Penske Truck Leasing Co LP",
"Weyerhaeuser Company Ltd",
"EPR Properties",
"BlueLinx Holdings Inc",
"Hyco International Inc",
"Johns Manville Corp",
"Extendicare Inc",
"Sevenson Environmental Services Inc",
"Comcast Cable Communications Inc.",
"Appalachian Power",
"Ryerson Tull Inc-Old",
"KT Corporation",
"West Marine Inc",
"Swift Transportation Co",
"Standard Register Co (The)",
"Public Service Company of New Hampshire",
"Regional Brands Inc",
"Connecticut Light & Power Co",
"Sure Energy Inc",
"Loomis Fargo & Co",
"Garda World Security Corp",
"Petro USA Inc",
"Gas Natural Inc",
"Verizon Inc/PA",
"Consolidated Container Co LLC",
"Vanguard Car Rental Group Inc",
"Tube City IMS Corp",
"Data Systems Inc",
"G&K Services Inc",
"West Energy Ltd",
"Noble Environmental Power Inc",
"Chrysler Group LLC",
"Bea Systems Inc",
"RAE Systems Inc.

# check downloaded glassdoor reviews data

In [54]:
df = pd.read_csv(r'D:\Users\wangy\Documents\temp\Glassdoor岗位评价.csv')

C:\Users\wangy\AppData\Local\Temp\ipykernel_10056\1378211149.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'D:\Users\wangy\Documents\temp\Glassdoor岗位评价.csv')


In [12]:
gd_columns = [
    'GD_JobTitle', 'glassdoor_web', 'GD_CompanyName', 'glassdoor_id', 'GD_ReviewDate',
    'GD_Rating', 'GD_ReviewTitle', 'GD_ReviewerStatus', 'GD_Pros', 'GD_Cons', 'GD_Advice',
    'GD_Recommend', 'GD_CEOSupport', 'GD_Outlook', 'GD_CareerOpp', 'GD_CompBenefits',
    'GD_Management', 'GD_WorkLife', 'GD_CultureValues', 'GD_Diversity', 'GD_Index'
]

In [62]:
sample_gd_df = pd.read_csv(r'D:\Users\wangy\Documents\temp\Glassdoor岗位评价.csv', nrows=1000, encoding='utf-8-sig')
sample_gd_df.columns = gd_columns
sample_gd_df.to_excel(os.path.join(const.TEMP_PATH, 'glassdoor_review_data_sample.xlsx'), index=False)

In [7]:
df.head()

,岗位名称,Glassdoor公司链接,公司名称,公司代码,评论日期,评分,评论标题,评论者状态,正评,负评,...,推荐公司,CEO支持率,公司展望,职业机会,薪酬与福利,高级管理层,工作与生活平衡,文化与价值观,多元化与包容性,索引编号
0,Manager Design,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,Baja Steel and Fence,5462645,2022-11-19,5.0,Good,"Current Employee, more than 10 years",Knowledge gain of complete project,Financial growth and personal growth,...,v,o,v,3.0,3.0,3.0,3.0,3.0,3.0,NaN
1,Anonymous Employee,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,Baja Steel and Fence,5462645,2022-01-29,4.0,Good,"Former Employee, less than 1 year","Good work,good work , flexible, support","Good,work, flexible,good support, good team work",...,v,o,o,4.0,4.0,4.0,4.0,4.0,4.0,NaN
2,Production Engineer,Reviews/Baja-Steel-and-Fence-Reviews-E5462645.htm,Baja Steel and Fence,5462645,2021-08-12,4.0,"Supervising the manufacturing the processes, e...","Current Employee, more than 1 year",This company is a best opportunity for me to l...,"Monthly Target work,Maintain production schedu...",...,v,o,v,2.0,3.0,2.0,2.0,2.0,2.0,NaN
3,Senior Account Executive,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames,5247,2020-09-24,1.0,terrible,"Current Employee, more than 1 year",I wish there were some to list,too many to list here,...,x,x,x,1.0,3.0,1.0,3.0,1.0,NaN,NaN
4,Assistant Manager,https://www.glassdoor.com/Reviews/Calgary-Flam...,Calgary Flames,5247,2023-03-25,4.0,"It could be so good, but it isn’t","Current Employee, more than 3 years",Fast Paced. Endless challenges. Inclusive envi...,The biggest perk of the job provides no value ...,...,o,o,o,3.0,3.0,3.0,1.0,4.0,5.0,NaN


In [8]:
df.keys()

Index(['岗位名称', 'Glassdoor公司链接', '公司名称', '公司代码', '评论日期', '评分', '评论标题', '评论者状态',
       '正评', '负评', '建议', '推荐公司', 'CEO支持率', '公司展望', '职业机会', '薪酬与福利', '高级管理层',
       '工作与生活平衡', '文化与价值观', '多元化与包容性', '索引编号'],
      dtype='object')

In [5]:
def extract_glassdoor_company_id(url):
    if not isinstance(url, str) or pd.isna(url):
        return None
    match = re.search(r'-E(\d+)\.htm', url)
    if match:
        return int(match.group(1))
    return None

In [6]:
glassdoor_url = pd.read_excel(os.path.join(const.TEMP_PATH, '20250625_ue_with_glassdoor_web_fill_miss.xlsx'))
glassdoor_url['glassdoor_id'] = glassdoor_url['glassdoor_web'].apply(extract_glassdoor_company_id)

In [13]:
df[df['索引编号'].notnull()]

,公司代码,评论日期,评分,评论标题,索引编号
292308,325091,2020-12-18,5.0,great work envrionment,0.0
292309,325091,2021-08-03,2.0,No growth,1.0
292310,325091,2021-02-16,4.0,Customer Service Representative,2.0
292311,325091,2020-09-13,5.0,Great place,3.0
292312,325091,2020-08-21,5.0,Great company,4.0
...,...,...,...,...,...
9729485,320958,2021-09-16,5.0,Good work life environment,5.0
9729486,320958,2021-09-13,4.0,Robert bosch,6.0
9729487,320958,2021-09-28,3.0,Good company,7.0
9729488,320958,2021-09-03,5.0,Great place to work,8.0


In [14]:
df.columns = gd_columns

In [22]:
df2 = df.drop_duplicates()

In [23]:
df2.shape

(9634034, 21)

In [24]:
df.shape

(9901889, 21)

In [25]:
df2.to_csv(r'D:\Users\wangy\Documents\temp\Glassdoor_Comments.csv', index=False)


In [26]:
glassdoor_with_comments = glassdoor_url.merge(df2.drop(['glassdoor_web'], axis=1), on=['glassdoor_id'], how='inner')
glassdoor_with_comments['gvkey'].drop_duplicates().shape

(333,)

In [29]:
glassdoor_with_comments.to_pickle(os.path.join(const.TEMP_PATH, '20250630_glassdoor_comments_with_union_gvkey.pkl'))

In [53]:
ue_gvkey_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '1950_2025_union_election_gvkey3.pkl'))
ue_gvkey_df = ue_gvkey_df.loc[ue_gvkey_df[const.ELECTION_YEAR] >= 2008]
glassdoor_with_comments['GD_ReviewDate'] = pd.to_datetime(glassdoor_with_comments['GD_ReviewDate'])
glassdoor_with_comments['GD_ReviewYear'] = glassdoor_with_comments['GD_ReviewDate'].dt.year

In [56]:
ue_gvkey_df = ue_gvkey_df.loc[ue_gvkey_df[const.GVKEY].isin(set(glassdoor_with_comments[const.GVKEY]))]

In [55]:
ue_gvkey_df[const.VOTE_SHARE] = ue_gvkey_df[const.NUM_VOTE_FOR] / ue_gvkey_df[const.NUM_VOTES]
ue_gvkey_df[const.MARGIN_OF_VICTORY] = ue_gvkey_df[const.VOTE_SHARE] - 0.5
ue_gvkey_df[const.ABS_MARGIN_SHARE] = ue_gvkey_df[const.MARGIN_OF_VICTORY].apply(abs)

In [61]:
ue_gvkey_df.shape

(1680, 18)

In [57]:
ue_gvkey_df.sort_values(by=[const.GVKEY, const.ELECTION_YEAR, const.ELECTION_MONTH, const.ELECTION_DAY], ascending=[True, True, True, True], inplace=True)
ue_gvkey_df2 = ue_gvkey_df[ue_gvkey_df[const.NUM_VOTES] >= 10]
ue_gvkey_df2.shape

(1158, 18)

In [58]:
ue_gvkey_df2.drop_duplicates(subset=[const.GVKEY, const.ELECTION_YEAR], keep='first').shape

(665, 18)

In [59]:
unique_election = ue_gvkey_df2.drop_duplicates(subset=[const.GVKEY, const.ELECTION_YEAR], keep='first')
unique_election.loc[unique_election[const.ABS_MARGIN_SHARE] < 0.2].shape

(352, 18)

In [60]:
unique_election.loc[unique_election[const.ABS_MARGIN_SHARE] < 0.1].shape

(184, 18)

In [47]:
unique_election[const.NUM_VOTES]

222543      4.0
221930    170.0
225501     15.0
217003      8.0
238178    305.0
          ...  
224822     18.0
237074      6.0
232299      3.0
238485     34.0
229745     42.0
Name: number_of_votes, Length: 844, dtype: float64